In [0]:
import requests
from datetime import datetime
from zoneinfo import ZoneInfo


url = "https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_status.json"

response = requests.get(url, timeout=15)
response.raise_for_status()
retrieved_ts = datetime.now(ZoneInfo("America/New_York"))

data = response.json()
stations = data["data"]["stations"]
print(f"Stations retrieved: {len(stations)}")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    ArrayType
)
from pyspark.sql.functions import lit


# Schema for each vehicle type within a station
vehicle_type_schema = StructType([
    StructField("vehicle_type_id", StringType(), True),
    StructField("count", IntegerType(), True)
])

# Schema for each station_status record
station_status_schema = StructType([
    StructField("station_id", StringType(), True),
    StructField("num_bikes_available", IntegerType(), True),
    StructField("num_bikes_disabled", IntegerType(), True),
    StructField("num_docks_available", IntegerType(), True),
    StructField("num_docks_disabled", IntegerType(), True),
    StructField("is_installed", IntegerType(), True),
    StructField("is_renting", IntegerType(), True),
    StructField("is_returning", IntegerType(), True),
    StructField("last_reported", LongType(), True),
    StructField(
        "vehicle_types_available",
        ArrayType(vehicle_type_schema),
        True
    ),
    StructField("num_ebikes_available", IntegerType(), True),
    StructField("num_scooters_available", IntegerType(), True),
    StructField("num_scooters_unavailable", IntegerType(), True)
])

df = spark.createDataFrame(
    stations,
    schema=station_status_schema
)

df = df.withColumn("retrieved_ts", lit(retrieved_ts))

df.printSchema()
display(df.limit(3))

In [0]:
df = df.withColumn("scheduled_ts", lit(retrieved_ts))

In [0]:
table_name = "workspace.default.citibike_station_status_history"

df.limit(0).write \
    .format("delta") \
    .mode("ignore") \
    .saveAsTable(table_name)

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, table_name)

(
    target.alias("target")
    .merge(
        df.alias("source"),
        """
        target.station_id = source.station_id
        AND target.scheduled_ts = source.scheduled_ts
        """
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
spark.table(table_name).count()